In [7]:
import torch
from torch.utils.data import DataLoader
from seqeval.metrics import classification_report
from transformers import AutoTokenizer

from evaluate import evaluate_bert_softmax
from models.matscibert_softmax import MatSciBertSoftmaxNER
from utils.bert.data_utils import read_conll_2, NERDataset, build_collate_fn

In [8]:
MODEL_NAME = 'm3rg-iitd/matscibert'
MAX_LENGTH = 128
BATCH_SIZE = 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_PATH = '../data/matscholar/test.txt'
MODEL_PATH = '../models/matscibert_softmax.pt'

In [9]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

tag2idx = checkpoint['tag2idx']
idx2tag = {v: k for k, v in tag2idx.items()}

test_sentence, test_tags = read_conll_2(TEST_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

test_dataset = NERDataset(test_sentence, test_tags)

collate_fn = build_collate_fn(
    tokenizer=tokenizer,
    label2id=tag2idx,
    max_length=MAX_LENGTH
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

model = MatSciBertSoftmaxNER(
    model_name=MODEL_NAME,
    num_labels=len(tag2idx),
    id2label=idx2tag,
    label2id=tag2idx,
).to(DEVICE)

model.load_state_dict(checkpoint['model'])

test_loss, test_p, test_r, test_f1, y_true, y_pred = evaluate_bert_softmax(
    model, test_loader, idx2tag, DEVICE
)

print(f"\n========== Test Result ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Precision: {test_p:.4f}")
print(f"Test Recall: {test_r:.4f}")
print(f"Test F1: {test_f1:.4f}")

print(f"\n========== Test Detailed Report ==========")
print(classification_report(y_true, y_pred, digits=4))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: m3rg-iitd/matscibert
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



========== Test Result ==========
Test Loss: 0.2436
Test Precision: 0.8615
Test Recall: 0.8878
Test F1: 0.8744

========== Test Detailed Report ==========
              precision    recall  f1-score   support

         APL     0.7766    0.7766    0.7766        94
         CMT     0.8639    0.8967    0.8800       184
         DSC     0.9158    0.9204    0.9181       402
         MAT     0.8996    0.9370    0.9179       698
         PRO     0.8028    0.8196    0.8112       621
         SMT     0.8511    0.9009    0.8753       222
         SPL     0.8085    0.9048    0.8539        42

   micro avg     0.8615    0.8878    0.8744      2263
   macro avg     0.8455    0.8794    0.8619      2263
weighted avg     0.8615    0.8878    0.8743      2263

